<a href="https://colab.research.google.com/github/MahathiKadiyala/24B11DS084_DriveReady_OPC_2026/blob/main/Day-5/Agentic_AI_Day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%pip install -q -U langchain langchain-groq langgraph chromadb gspread google-auth pydantic
%pip install -q --upgrade --force-reinstall opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions

print("Package installation completed.")
print("Next: Runtime → Restart session, then start again from the imports cell below.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-genai 2.12.1 requires google-auth[requests]<2.56.0,>=2.48.1, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.12.1 requires google-auth[requests]<2.56.0,>=2.48.1, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39

In [1]:
import os
import json
import uuid
from datetime import datetime, timezone
from typing import Optional

import pandas as pd
import chromadb
import gspread

from google.colab import userdata
from google.colab import auth
from google.auth import default

from pydantic import BaseModel, Field, ValidationError
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

print("Imports completed successfully.")
print("ChromaDB:", chromadb.__version__)

Imports completed successfully.
ChromaDB: 1.5.9


In [2]:
api_key = userdata.get("agentic_ai_GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY was not found. Add it in Colab Secrets and enable notebook access."
    )

os.environ["GROQ_API_KEY"] = api_key

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq model connected.")

Groq model connected.


In [3]:
SYSTEM_DAY4 = """
You are VoltCare AI, a helpful EV customer and service enquiry assistant.

Your responsibilities:
- Explain the services listed in the approved service catalogue.
- Use the price lookup tool when asked about prices.
- Use the service catalogue tool to check whether a service is listed.
- Collect customer details only when the customer wants to submit an enquiry.
- Use the enquiry logging tool only after the customer has provided the required details
  and clearly agreed to submit the enquiry.

Important limitations:
- The service catalogue check is NOT a live appointment-slot or technician availability check.
- Do not claim an appointment is booked or confirmed.
- Do not invent service-centre addresses, opening hours, warranties, free replacements,
  discounts, battery guarantees, or company policies.
- Prices in this workshop are illustrative estimates, not verified quotations. Clearly
  disclose this whenever you give a price.
- If a fact is not in the tools or knowledge base, say it is not confirmed and recommend
  that the VoltCare team verify it.
- Never claim an enquiry was saved unless the logging tool confirms successful saving.
- Ask for missing details one at a time. Do not request unnecessary sensitive information.
- Be polite, concise, and clear.
"""

print("Day 4 system instructions ready.")

Day 4 system instructions ready.


In [4]:
SERVICE_CATALOGUE = {
    "general servicing": {
        "display_name": "General EV servicing",
        "illustrative_price_inr": 1500
    },
    "battery check": {
        "display_name": "EV battery check",
        "illustrative_price_inr": 800
    },
    "tyre check": {
        "display_name": "Tyre check",
        "illustrative_price_inr": 300
    }
}

def normalize_service_name(service_name: str) -> str:
    value = service_name.strip().lower().replace("_", " ").replace("-", " ")
    aliases = {
        "general service": "general servicing",
        "ev servicing": "general servicing",
        "battery inspection": "battery check",
        "ev battery check": "battery check",
        "tyre inspection": "tyre check",
        "tire check": "tyre check",
    }
    return aliases.get(value, value)

print("Service catalogue entries:", len(SERVICE_CATALOGUE))
for key, item in SERVICE_CATALOGUE.items():
    print("-", item["display_name"], "— illustrative ₹", item["illustrative_price_inr"])

Service catalogue entries: 3
- General EV servicing — illustrative ₹ 1500
- EV battery check — illustrative ₹ 800
- Tyre check — illustrative ₹ 300


In [5]:
@tool
def get_service_price(service_name: str) -> str:
    """Look up the illustrative training estimate for a VoltCare EV service."""
    key = normalize_service_name(service_name)

    if key not in SERVICE_CATALOGUE:
        return (
            f"No price is configured for '{service_name}'. "
            "The VoltCare team must confirm the service and quotation."
        )

    item = SERVICE_CATALOGUE[key]
    return (
        f"{item['display_name']}: illustrative estimate ₹"
        f"{item['illustrative_price_inr']}. This is a workshop example, "
        "not a verified VoltCare quotation; the team must confirm the final price."
    )

print(get_service_price.invoke({"service_name": "battery check"}))

EV battery check: illustrative estimate ₹800. This is a workshop example, not a verified VoltCare quotation; the team must confirm the final price.


In [6]:
@tool
def check_service_supported(service_name: str) -> str:
    """Check whether a service is listed in the configured catalogue; this is not live slot availability."""
    key = normalize_service_name(service_name)

    if key in SERVICE_CATALOGUE:
        return (
            f"'{SERVICE_CATALOGUE[key]['display_name']}' is listed in the current "
            "workshop service catalogue. This does not confirm a booking, service "
            "centre, technician, or appointment slot."
        )

    return (
        f"'{service_name}' is not listed in the current catalogue. "
        "The VoltCare team must confirm whether it can be supported."
    )

print(check_service_supported.invoke({"service_name": "battery check"}))

'EV battery check' is listed in the current workshop service catalogue. This does not confirm a booking, service centre, technician, or appointment slot.


In [7]:
class EVServiceEnquiry(BaseModel):
    customer_name: str = Field(min_length=2, max_length=100)
    vehicle_model: str = Field(min_length=2, max_length=100)
    service_type: str = Field(min_length=2, max_length=100)
    contact_number: str = Field(min_length=7, max_length=20)
    preferred_date: str = Field(min_length=2, max_length=50)
    issue_description: str = Field(min_length=3, max_length=1000)

print("Pydantic enquiry model created.")
print("Fields:", list(EVServiceEnquiry.model_fields.keys()))

Pydantic enquiry model created.
Fields: ['customer_name', 'vehicle_model', 'service_type', 'contact_number', 'preferred_date', 'issue_description']


In [8]:
gc = None
sheet = None
worksheet = None
SHEETS_READY = False

try:
    auth.authenticate_user()
    credentials, project_id = default()
    gc = gspread.authorize(credentials)
    sheet = gc.open("VoltCare_AI_Enquiries")
    worksheet = sheet.sheet1
    SHEETS_READY = True

    print("Connected to:", sheet.title)
    print("Worksheet:", worksheet.title)
    print("Existing header row:", worksheet.row_values(1))

except Exception as exc:
    print("Google Sheets connection is not ready.")
    print("Reason:", str(exc)[:500])
    print("You can continue; the logging tool will not claim an enquiry was saved.")

Connected to: VoltCare_AI_Enquiries
Worksheet: Sheet1
Existing header row: ['Customer Name', 'EV Model', 'Service Type', 'Issue Description', 'Contact Number', 'Preferred Date']


In [39]:
EXPECTED_HEADERS = [
    "customer_name",
    "vehicle_model",
    "service_type",
    "contact_number",
    "preferred_date",
    "issue_description"
]

HEADER_ALIASES = {
    "customer_name": {
        "customer_name", "customer name", "name"
    },
    "vehicle_model": {
        "vehicle_model", "vehicle model", "vehicle", "ev model"
    },
    "service_type": {
        "service_type", "service type", "service"
    },
    "contact_number": {
        "contact_number", "contact number", "phone", "phone number"
    },
    "preferred_date": {
        "preferred_date", "preferred date", "date"
    },
    "issue_description": {
        "issue_description", "issue description", "problem", "issue"
    },
}

def get_sheet_column_map():
    if not SHEETS_READY or worksheet is None:
        raise RuntimeError("Google Sheets is not connected.")

    headers = worksheet.row_values(1)

    if not headers:
        worksheet.append_row(EXPECTED_HEADERS)
        headers = EXPECTED_HEADERS.copy()

    normalized_headers = [h.strip().lower() for h in headers]
    column_map = {}

    for field_name, aliases in HEADER_ALIASES.items():
        for index, header in enumerate(normalized_headers, start=1):
            if header in aliases:
                column_map[field_name] = index
                break

    missing = [name for name in EXPECTED_HEADERS if name not in column_map]
    if missing:
        raise RuntimeError(
            "The existing sheet is missing required columns: "
            + ", ".join(missing)
            + ". No row was written. Review the header row before changing it."
        )

    return column_map

if SHEETS_READY:
    try:
        sheet_column_map = get_sheet_column_map()
        print("Google Sheets headers are compatible.")
        print("Column mapping:", sheet_column_map)
    except Exception as exc:
        sheet_column_map = None
        print("Sheet is connected, but enquiry writing is not enabled:", exc)
else:
    sheet_column_map = None
    print("Sheet mapping skipped because Google Sheets is not connected.")

Google Sheets headers are compatible.
Column mapping: {'customer_name': 1, 'vehicle_model': 2, 'service_type': 3, 'contact_number': 5, 'preferred_date': 6, 'issue_description': 4}


In [10]:
@tool
def log_service_enquiry(
    customer_name: str,
    vehicle_model: str,
    service_type: str,
    contact_number: str,
    preferred_date: str,
    issue_description: str
) -> str:
    """Validate and save a customer service enquiry to the connected VoltCare Google Sheet."""

    try:
        enquiry = EVServiceEnquiry(
            customer_name=customer_name,
            vehicle_model=vehicle_model,
            service_type=service_type,
            contact_number=contact_number,
            preferred_date=preferred_date,
            issue_description=issue_description
        )
    except ValidationError as exc:
        return "Enquiry details are incomplete or invalid. Please correct them: " + str(exc)

    if not SHEETS_READY or worksheet is None:
        return (
            "The enquiry has NOT been saved because Google Sheets is not connected. "
            "Please try again after the connection is restored."
        )

    try:
        column_map = get_sheet_column_map()
        values = enquiry.model_dump()
        headers = worksheet.row_values(1)
        row = [""] * len(headers)

        for field_name, column_index in column_map.items():
            row[column_index - 1] = values[field_name]

        worksheet.append_row(row, value_input_option="USER_ENTERED")

        return (
            "The service enquiry was successfully saved to the VoltCare enquiries sheet. "
            "This is an enquiry only; no appointment has been booked or confirmed."
        )

    except Exception as exc:
        return (
            "The enquiry was NOT confirmed as saved. Google Sheets returned an error: "
            + str(exc)[:300]
        )

print("Enquiry logging tool created. No test row has been added.")

Enquiry logging tool created. No test row has been added.


In [12]:
day4_tools = [
    get_service_price,
    check_service_supported,
    log_service_enquiry
]

voltcare_agent = create_agent(
    model=llm,
    tools=day4_tools,
    system_prompt=SYSTEM_DAY4
)

print("Day 4 VoltCare agent created.")

Day 4 VoltCare agent created.


In [13]:
day4_test_questions = [
    "What is the illustrative price of a battery check?",
    "Do you list tyre checks?",
    "Can you confirm an appointment slot for tomorrow?"
]

for question in day4_test_questions:
    print("\nCUSTOMER:", question)
    result = voltcare_agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    print("VOLTCARE:", result["messages"][-1].content)


CUSTOMER: What is the illustrative price of a battery check?
VOLTCARE: The illustrative estimate for a **battery check** is **₹800**.  
Please note that this is only an example figure; the final price will be confirmed by the VoltCare team.

CUSTOMER: Do you list tyre checks?
VOLTCARE: Yes – a **tyre check** is listed in our service catalogue.  
It’s a routine inspection of tyre pressure, tread depth, and overall condition to help keep your EV safe and efficient.  

If you’d like a price estimate or to schedule an enquiry, just let me know!

CUSTOMER: Can you confirm an appointment slot for tomorrow?
VOLTCARE: I’m sorry, but I don’t have the ability to check or confirm appointment slots. I can help you with information about our services, give you an illustrative price estimate, or guide you through setting up an enquiry if you’d like to book a time. Let me know how I can assist!


In [16]:
RUN_REAL_SHEETS_TEST = False
if RUN_REAL_SHEETS_TEST:
    test_result = log_service_enquiry.invoke({
        "customer_name": "Workshop Test Customer",
        "vehicle_model": "Test EV",
        "service_type": "Battery check",
        "contact_number": "0000000000",
        "preferred_date": "Test date - remove manually",
        "issue_description": "Clearly labelled workshop test row; remove manually."
    })
    print(test_result)
else:
    print("Skipped. No test row was written to Google Sheets.")

Skipped. No test row was written to Google Sheets.


In [17]:
import chromadb
import pandas as pd

from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

print("Day 5 libraries imported successfully.")

Day 5 libraries imported successfully.


In [18]:
voltcare_facts = [
    "VoltCare AI is a customer and service enquiry assistant for electric vehicle services.",
    "The assistant can answer questions using the business facts and service catalogue provided to it.",
    "The example service catalogue includes general EV servicing.",
    "The example service catalogue includes EV battery checks.",
    "The example service catalogue includes tyre checks.",
    "The illustrative training estimate for general EV servicing is ₹1,500.",
    "The illustrative training estimate for an EV battery check is ₹800.",
    "The illustrative training estimate for a tyre check is ₹300.",
    "All prices in this workshop are examples and are not verified company quotations.",
    "The final service price must be confirmed by the VoltCare team.",
    "Customers may ask the assistant to submit a service enquiry.",
    "An enquiry may include customer name, vehicle model, requested service, contact number, preferred date, and issue description.",
    "The assistant validates enquiry details using a Pydantic model.",
    "The enquiry logging tool can save an enquiry only when Google Sheets is connected and the write succeeds.",
    "A saved enquiry is not an appointment booking or booking confirmation.",
    "The service catalogue tool checks whether a service is listed; it does not check live appointment slots.",
    "The assistant does not have live technician schedules or live service-centre availability.",
    "No warranty, free battery replacement, discount, or guarantee should be promised unless verified policy information is supplied.",
    "If a business fact is missing, the assistant must say it is not confirmed and ask the VoltCare team to verify it.",
    "The assistant must not invent service-centre addresses, opening hours, vehicle telemetry, or repair outcomes."
]

assert len(voltcare_facts) == 20
print("VoltCare facts ready:", len(voltcare_facts))

VoltCare facts ready: 20


In [19]:
chroma_client = chromadb.Client()

voltcare_collection = chroma_client.get_or_create_collection(
    name="voltcare_business_knowledge",
    metadata={"description": "VoltCare AI business facts and service limitations"}
)

print("ChromaDB collection created.")

ChromaDB collection created.


In [20]:
fact_ids = [f"voltcare_fact_{i:02d}" for i in range(1, len(voltcare_facts) + 1)]

voltcare_collection.upsert(
    ids=fact_ids,
    documents=voltcare_facts
)

print("Knowledge stored successfully.")
print("Total documents:", voltcare_collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 95.6MiB/s]


Knowledge stored successfully.
Total documents: 20


In [21]:
stored_facts = voltcare_collection.get()

for fact_id, document in zip(stored_facts["ids"], stored_facts["documents"]):
    print(f"{fact_id}: {document}")

print("\nStored document count:", len(stored_facts["ids"]))

voltcare_fact_01: VoltCare AI is a customer and service enquiry assistant for electric vehicle services.
voltcare_fact_02: The assistant can answer questions using the business facts and service catalogue provided to it.
voltcare_fact_03: The example service catalogue includes general EV servicing.
voltcare_fact_04: The example service catalogue includes EV battery checks.
voltcare_fact_05: The example service catalogue includes tyre checks.
voltcare_fact_06: The illustrative training estimate for general EV servicing is ₹1,500.
voltcare_fact_07: The illustrative training estimate for an EV battery check is ₹800.
voltcare_fact_08: The illustrative training estimate for a tyre check is ₹300.
voltcare_fact_09: All prices in this workshop are examples and are not verified company quotations.
voltcare_fact_10: The final service price must be confirmed by the VoltCare team.
voltcare_fact_11: Customers may ask the assistant to submit a service enquiry.
voltcare_fact_12: An enquiry may includ

In [22]:
test_search = voltcare_collection.query(
    query_texts=["What is the illustrative price of an EV battery check?"],
    n_results=3
)

for document in test_search["documents"][0]:
    print("-", document)

- The illustrative training estimate for an EV battery check is ₹800.
- The example service catalogue includes EV battery checks.
- The illustrative training estimate for general EV servicing is ₹1,500.


In [23]:
@tool
def search_ev_knowledge(question: str) -> str:
    """
    Search the VoltCare AI business knowledge base.

    Topics covered:
    - VoltCare overview and assistant scope
    - General EV servicing, battery checks, and tyre checks
    - Illustrative workshop prices and price disclaimers
    - Customer enquiry fields and Pydantic validation
    - Google Sheets enquiry saving limitations
    - Difference between an enquiry and an appointment booking
    - Service catalogue support versus live appointment availability
    - Warranty and policy information limitations
    """
    results = voltcare_collection.query(
        query_texts=[question],
        n_results=4
    )

    documents = results.get("documents", [[]])[0]

    if not documents:
        return "No relevant information was found in the VoltCare knowledge base."

    return "\n".join(f"- {document}" for document in documents)

print(search_ev_knowledge.invoke({
    "question": "Does VoltCare guarantee free battery replacement?"
}))

- No warranty, free battery replacement, discount, or guarantee should be promised unless verified policy information is supplied.
- The example service catalogue includes EV battery checks.
- VoltCare AI is a customer and service enquiry assistant for electric vehicle services.
- The final service price must be confirmed by the VoltCare team.


In [24]:
@tool
def get_service_price_day5(service_name: str) -> str:
    """Retrieve the configured illustrative estimate for a VoltCare service."""
    return get_service_price.invoke({"service_name": service_name})


@tool
def check_service_supported_day5(service_name: str) -> str:
    """Check catalogue listing only; this does not check live appointment availability."""
    return check_service_supported.invoke({"service_name": service_name})


@tool
def log_service_enquiry_day5(
    customer_name: str,
    vehicle_model: str,
    service_type: str,
    contact_number: str,
    preferred_date: str,
    issue_description: str
) -> str:
    """Validate and save a customer enquiry to the connected Google Sheet when available."""
    return log_service_enquiry.invoke({
        "customer_name": customer_name,
        "vehicle_model": vehicle_model,
        "service_type": service_type,
        "contact_number": contact_number,
        "preferred_date": preferred_date,
        "issue_description": issue_description
    })

print("Day 5 service tool wrappers created.")

Day 5 service tool wrappers created.


In [40]:
SYSTEM_DAY5 = """
You are VoltCare AI, a helpful EV customer and service enquiry assistant.

GROUNDING RULES:
- Search the VoltCare knowledge base for business facts and policy questions.
- Answer business questions only using information supported by the knowledge base.
- If the knowledge base does not contain the answer, clearly say:
  "This information is not currently confirmed. Please check with the VoltCare team."
- Never invent missing information or make assumptions.

TOOLS:
- Use the price tool for service prices.
- Always explain that workshop prices are illustrative estimates,
  not verified company quotations.
- Use the service support tool only to check catalogue listings.
- Use the enquiry logging tool only when the customer explicitly
  requests an enquiry and provides all required details.

LIMITATIONS:
- Catalogue support does not mean appointment availability.
- Never claim an appointment is booked or confirmed.
- Do not invent warranty terms, free replacements, discounts,
  service-centre locations, opening hours, booking portals,
  technician schedules, live vehicle data or repair outcomes.
- Never claim an enquiry was saved unless Google Sheets confirms success.

MEMORY:
- Maintain context within the current conversation thread.
- Never mix information belonging to different thread IDs.

RESPONSE STYLE:
- Be polite, concise and helpful.
- Ask for missing enquiry details one at a time.
- Be transparent whenever information is unavailable.
"""
print("Day 5 system prompt ready.")

Day 5 system prompt ready.


In [26]:
day5_tools = [
    get_service_price_day5,
    check_service_supported_day5,
    log_service_enquiry_day5,
    search_ev_knowledge
]

memory = InMemorySaver()

print("Day 5 tools:", [tool.name for tool in day5_tools])
print("In-memory conversation saver created.")

Day 5 tools: ['get_service_price_day5', 'check_service_supported_day5', 'log_service_enquiry_day5', 'search_ev_knowledge']
In-memory conversation saver created.


In [27]:
voltcare_agent_day5 = create_agent(
    model=llm,
    tools=day5_tools,
    system_prompt=SYSTEM_DAY5,
    checkpointer=memory
)

print("VoltCare Day 5 RAG + memory agent created.")

VoltCare Day 5 RAG + memory agent created.


In [28]:
def chat_day5(thread_id: str, message: str) -> str:
    if not thread_id.strip():
        raise ValueError("thread_id cannot be empty.")

    result = voltcare_agent_day5.invoke(
        {"messages": [{"role": "user", "content": message}]},
        config={"configurable": {"thread_id": thread_id}}
    )

    return result["messages"][-1].content

print("Chat helper ready.")

Chat helper ready.


In [29]:
print(chat_day5(
    "customer_001",
    "What is the price of an EV battery check?"
))

The illustrative estimate for an EV battery check is **₹800**.  
Please note that this is a workshop example and not a verified quotation; the VoltCare team should confirm the final price.


In [30]:
print(chat_day5(
    "customer_001",
    "Do you list tyre checks? Does that mean an appointment is available today?"
))

Yes, a **tyre check** is listed in our service catalogue.  
However, that listing does **not** indicate that an appointment slot is currently available. To find out if a time today (or any other date) works for you, please contact the VoltCare team directly or use our online booking portal.


In [31]:
print(chat_day5(
    "customer_rahul",
    "My name is Rahul and I drive a Tata Nexon EV."
))

print("\nFollow-up:")
print(chat_day5(
    "customer_rahul",
    "Which vehicle do I drive?"
))

Hello Rahul! How can I help you with your Tata Nexon EV today?

Follow-up:
You mentioned earlier that you drive a **Tata Nexon EV**. Is that correct?


In [32]:
print(chat_day5(
    "customer_priya",
    "My name is Priya and I drive an MG ZS EV."
))

print("\nAsk the second thread about its vehicle:")
print(chat_day5(
    "customer_priya",
    "Which vehicle do I drive?"
))

print("\nAsk the first thread again:")
print(chat_day5(
    "customer_rahul",
    "Which vehicle did I tell you I drive?"
))

Hello Priya! How can I assist you with your MG ZS EV today?

Ask the second thread about its vehicle:
You drive an MG ZS EV.

Ask the first thread again:
You told me you drive a **Tata Nexon EV**.


In [33]:
warranty_answer = chat_day5(
    "customer_warranty_test",
    "Does VoltCare provide free battery replacement with a 10-year warranty?"
)

print(warranty_answer)

I’m not aware of any policy that offers a free battery replacement with a 10‑year warranty. That information isn’t in the VoltCare knowledge base, so I can’t confirm it. It’s best to check directly with the VoltCare team or review the official warranty documentation for your specific vehicle.


In [34]:
for item in day5_tools:
    print("\nTOOL:", item.name)
    print("DESCRIPTION:", item.description)


TOOL: get_service_price_day5
DESCRIPTION: Retrieve the configured illustrative estimate for a VoltCare service.

TOOL: check_service_supported_day5
DESCRIPTION: Check catalogue listing only; this does not check live appointment availability.

TOOL: log_service_enquiry_day5
DESCRIPTION: Validate and save a customer enquiry to the connected Google Sheet when available.

TOOL: search_ev_knowledge
DESCRIPTION: Search the VoltCare AI business knowledge base.

Topics covered:
- VoltCare overview and assistant scope
- General EV servicing, battery checks, and tyre checks
- Illustrative workshop prices and price disclaimers
- Customer enquiry fields and Pydantic validation
- Google Sheets enquiry saving limitations
- Difference between an enquiry and an appointment booking
- Service catalogue support versus live appointment availability
- Warranty and policy information limitations


In [41]:
checks = []

def record_check(name, condition):
    passed = bool(condition)
    checks.append((name, passed))
    print(("PASS" if passed else "FAIL"), "-", name)

record_check(
    "Exactly 20 starter facts are defined",
    len(voltcare_facts) == 20
)

record_check(
    "ChromaDB contains 20 documents",
    voltcare_collection.count() == 20
)

record_check(
    "Fact IDs are unique",
    len(fact_ids) == len(set(fact_ids))
)

record_check(
    "Price tool returns battery estimate",
    "800" in get_service_price.invoke({"service_name": "battery check"})
)

record_check(
    "Unknown price is not invented",
    "no price is configured" in
    get_service_price.invoke(
        {"service_name": "unknown service"}
    ).lower()
)

record_check(
    "Catalogue support does not claim live slots",
    "does not confirm" in check_service_supported.invoke({"service_name": "battery check"}).lower()
)

record_check(
    "RAG search tool has a useful description",
    "knowledge base" in search_ev_knowledge.description.lower()
)

passed_count = sum(1 for _, passed in checks if passed)
total_count = len(checks)
print(f"\nDeterministic checks: {passed_count}/{total_count} passed "
      f"({passed_count / total_count * 100:.1f}%)")

PASS - Exactly 20 starter facts are defined
PASS - ChromaDB contains 20 documents
PASS - Fact IDs are unique
PASS - Price tool returns battery estimate
PASS - Unknown price is not invented
PASS - Catalogue support does not claim live slots
PASS - RAG search tool has a useful description

Deterministic checks: 7/7 passed (100.0%)


In [42]:
print("=" * 50)
print("VOLTCARE AI - DAY 5 FINAL SIGN-OFF")
print("=" * 50)

print(f"\nAutomated checks: {passed_count}/{total_count}")

pass_percentage = (passed_count / total_count) * 100

print(f"Pass percentage: {pass_percentage:.1f}%")

if pass_percentage >= 80:
    print("\nAutomated checks meet the 80% target.")
else:
    print("\nSome checks need correction before sign-off.")

print("\nManual verification required:")
print("1. Test a follow-up question in the same thread.")
print("2. Test two different customer thread IDs.")
print("3. Ask a question outside the knowledge base.")
print("4. Confirm the agent does not invent business policies.")
print("5. Verify Google Sheets enquiry saving, if connected.")

print("\nDay 5 implementation review completed.")

VOLTCARE AI - DAY 5 FINAL SIGN-OFF

Automated checks: 7/7
Pass percentage: 100.0%

Automated checks meet the 80% target.

Manual verification required:
1. Test a follow-up question in the same thread.
2. Test two different customer thread IDs.
3. Ask a question outside the knowledge base.
4. Confirm the agent does not invent business policies.
5. Verify Google Sheets enquiry saving, if connected.

Day 5 implementation review completed.
